In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
for root, dirs, files in os.walk(path):
    if len(files) > 0:
        print(root, "->", files[:5])
        break


In [ ]:
import os

root = "/kaggle/input/q3-stage3-2026/dataset"
print("Root folders:", os.listdir(root))

for name in os.listdir(root):
    p = os.path.join(root, name)
    if os.path.isdir(p):
        print(name, "->", os.listdir(p)[:5])


In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms as T

# Fixed paths (we confirmed them)
img_dir = "/kaggle/input/q3-stage3-2026/dataset/images"
msk_dir = "/kaggle/input/q3-stage3-2026/dataset/masks"

print("Image folder:", img_dir)
print("Mask folder :", msk_dir)

img_files = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
msk_files = sorted(glob.glob(os.path.join(msk_dir, "*.png")))

# Match by filename stem
img_map = {os.path.splitext(os.path.basename(f))[0]: f for f in img_files}
msk_map = {os.path.splitext(os.path.basename(f))[0]: f for f in msk_files}

common_keys = sorted(list(set(img_map.keys()) & set(msk_map.keys())))
img_files = [img_map[k] for k in common_keys]
msk_files = [msk_map[k] for k in common_keys]

print("Matched pairs:", len(img_files))

# Resize size
IMG_SIZE = 256
NUM_CLASSES = 8

# Image transform
img_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

class SUIMDataset(Dataset):
    def __init__(self, img_files, mask_files, img_transform=None, img_size=256):
        self.img_files = img_files
        self.mask_files = mask_files
        self.img_transform = img_transform
        self.img_size = img_size

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        mask_path = self.mask_fi




In [ ]:
!pip -q install segmentation-models-pytorch


In [ ]:
# TO DO
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
)

print("Model created:", type(model).__name__)




In [ ]:
# TO DO
import torch.nn as nn

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)  # (B,H,W) long

        optimizer.zero_grad()
        outputs = model(images)   # (B,C,H,W)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(loader)



In [ ]:
# TO DO
import os, glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms as T

img_dir = "/kaggle/input/q3-stage3-2026/dataset/images"
msk_dir = "/kaggle/input/q3-stage3-2026/dataset/masks"

img_files = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
msk_files = sorted(glob.glob(os.path.join(msk_dir, "*.png")))

img_map = {os.path.splitext(os.path.basename(f))[0]: f for f in img_files}
msk_map = {os.path.splitext(os.path.basename(f))[0]: f for f in msk_files}
keys = sorted(list(set(img_map.keys()) & set(msk_map.keys())))

img_files = [img_map[k] for k in keys]
msk_files = [msk_map[k] for k in keys]

print("Matched pairs:", len(img_files))

IMG_SIZE = 256
NUM_CLASSES = 8

class SUIMDataset(Dataset):
    def __init__(self, img_files, mask_files, img_size=256):
        self.img_files = img_files
        self.mask_files = mask_files
        self.img_size = img_size

        self.img_transform = T.Compose([
            T.Resize((self.img_size, self.img_size)),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        mask_path = self.mask_files[idx]

        # Image resize
        img = Image.open(img_path).convert("RGB")
        img = self.img_transform(img)

        # Mask resize (NEAREST!)
        mask = Image.open(mask_path).convert("L")
        mask = mask.resize((self.img_size, self.img_size), Image.NEAREST)
        mask = torch.from_numpy(np.array(mask)).long()

        mask = remap_mask(mask)

        return img, mask

dataset = SUIMDataset(img_files, msk_files, img_size=IMG_SIZE)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0)

# sanity check batch sizes
imgs, masks = next(iter(train_loader))
print("imgs:", imgs.shape, "masks:", masks.shape)
print("mask unique values sample:", torch.unique(masks[0])[:10])



In [ ]:
# TO DO
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

model.eval()

images, masks = next(iter(val_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)
    preds = torch.argmax(outputs, dim=1)

images = images.cpu()
masks = masks.cpu()
preds = preds.cpu()

n = min(3, images.size(0))
plt.figure(figsize=(12, 8))

for i in range(n):
    plt.subplot(n, 3, i*3 + 1)
    plt.imshow(images[i].permute(1,2,0))
    plt.axis("off")
    plt.title("Image")

    plt.subplot(n, 3, i*3 + 2)
    plt.imshow(masks[i].numpy(), cmap="tab20")
    plt.axis("off")
    plt.title("GT Mask")

    plt.subplot(n, 3, i*3 + 3)
    plt.imshow(preds[i].numpy(), cmap="tab20")
    plt.axis("off")
    plt.title("Pred Mask")

plt.tight_layout()
plt.show()

